# Synthetic Data Pipeline

This notebook runs the synthetic data generation pipeline.

### Before getting started:
- Ensure you have read `docs/*` and `README.md`
- Check that `params.py` and `config.py` are correct.
- Check the `call_LLM` and `red_write_data` functions in `processing.py` are correctly configured for your platform .
- Check that your input data exists and is correctly formatted.

First, import the required classes. 

In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [2]:
from src.data_generator import generate_patients, generate_admissions, generate_journeys, generate_clinical_notes, add_augmentations, save_final_outputs
from datetime import datetime
from config.params import PARAMS

/mnt/c/Users/Will Poulett/Documents/synthetic_clinical_notes/src/data_generator.py:405: SyntaxWarning: invalid escape sequence '\s'
  df["ChiefComplaintDescription"] = df["ChiefComplaintDescription"].str.split("\s+\(").str.get(0)
/mnt/c/Users/Will Poulett/Documents/synthetic_clinical_notes/src/data_generator.py:406: SyntaxWarning: invalid escape sequence '\s'
  df["DiagnosisDescription"] = df["DiagnosisDescription"].str.split("\s+\(").str.get(0)


In [3]:
# INSERT RUN NAME BELOW
# This will be saved in the journey dataset at the end of the notebook for evaluation purposes
run_name = "wp_add_orthopaedic_letter"
current_time = datetime.now()
version_tag = current_time.strftime("%Y-%m-%d") + f"/{run_name}"
print(f"Run name: {run_name}\nDate: {current_time}\nVersion tag: {version_tag}")

Run name: wp_add_orthopaedic_letter
Date: 2026-05-06 17:01:23.154440
Version tag: 2026-05-06/wp_add_orthopaedic_letter


## 1 - Get Patients Information and Admissions

- Generates a list of patients with corresponding admission reasons.


In [4]:
patient_generator = generate_patients()
patients = await patient_generator.run(return_output = True)
patient_generator.write_patients_to_dataset()

Generating 1 patients... DONE


In [5]:
admission_generator = generate_admissions()
admissions = await admission_generator.run(return_output = True)
admission_generator.write_admissions_to_dataset()

Generating 1 admissions... DONE


In [6]:
admissions[0][0]

'{"date": "2026-01-06", "time": "18:20", "method": "booked", "procedure": "Anterior cruciate ligament reconstruction of right knee", "allergies": "[]", "current_medications": "[\\"Combined oral contraceptive pill (ethinylestradiol/levonorgestrel) 1 tablet once daily\\",\\"Ibuprofen 400 mg orally up to three times daily as needed for knee pain (stopped 5 days pre-op as per instructions)\\",\\"Paracetamol 1 g orally up to four times daily as needed for pain\\"]", "past_medical_history": "[\\"Right anterior cruciate ligament tear following sports injury (2025)\\",\\"Recurrent right knee pain and instability\\",\\"Mild exercise-induced asthma in adolescence \\u2013 no regular inhaler use for several years\\",\\"No history of ischaemic heart disease, diabetes, hypertension or stroke\\",\\"No previous problems with anaesthesia\\"]", "admitting_consultant": "Dr. Elaine Brenda Parkinson (Consultant)", "ward": "Trauma and Orthopaedics Ward 3A (Female)", "specialty": "Trauma and Orthopaedics Ser

## 2 - Generating and Filtering Journeys

- Generates, validates, and adds details to patient journeys.
- Filtering removes any document types that are not listed as possible event types in `params.py`.

In [4]:
journey_generator = generate_journeys()
journeys = await journey_generator.run(return_outputs = True)
journey_generator.write_journeys_to_dataset()

Generating simple journeys... 
    
        You are an expert at generating synthetic patient clinical NHS pathways that are clinically realistic and will be used for AI evaluation.
        It is extremely important that these patient pathway details are high-quality, realistic and similar to real-world data. 
        
        # Instructions
        
        - Write a realistic series of documented events in a patient journey given the following admission details. You must include the entire sequence of events from start to finish.
        - The suggested event types you can use are ['ED event', 'ED review and hand-over', 'emergency admission', 'elective admission', 'post take ward round', 'general ward round', 'pre-op assessment', 'anaesthetics assessment', 'pre-op consent', 'pre-op checklist', 'operation', 'post-anaesthesia recovery', 'inter-specialty review', 'nursing', 'misc', 'therapy', 'orthopaedic referral'], although you may include any events you feel are realistic. This can i

In [6]:
journeys[0][0]

'{"event_type": "orthopaedic referral", "date": "2025-11-25", "time": "10:15", "staff": "[\\"Dr. Aya Kobayashi (Orthopaedic Surgeon)\\", \\"Dr. Pamela Mary Simmons (GP)\\"]", "details": "At the request of the patient\\u2019s GP, Dr. Pamela Mary Simmons, a formal written referral is generated to the Trauma and Orthopaedics Service for specialist review by orthopaedic surgeon Dr. Aya Kobayashi. The referral letter documents a 26\\u2011year\\u2011old female with a sports\\u2011related right anterior cruciate ligament (ACL) tear sustained earlier in 2025, who has completed several months of structured physiotherapy but continues to experience episodes of right knee giving\\u2011way, pain on pivoting, and difficulty returning to sport. Examination findings from the GP include a positive Lachman test and anterior drawer test on the right, with no gross varus/valgus instability and no locking, suggesting ongoing ACL insufficiency without obvious meniscal locking. The GP notes that plain right

In [1]:
import json
json.loads(journeys[0][0])

NameError: name 'journeys' is not defined

## 3 - Generate and Validate Clinical Notes

- Uses LLMs to generate clinical notes. 
- Validates each note using an LLM Judge. 

In [ ]:
clinical_note_generator = generate_clinical_notes()
notes = await clinical_note_generator.run(return_output = True)
clinical_note_generator.write_patient_documents_to_dataset()

Patient 0: Generating Notes... Error cleaning with LLM: No JSON pattern found in LLM output, returning raw_output: s


In [ ]:
import json
print(json.loads(notes[0][0])["Content"])

05/01/26

To: Orthopaedic Knee Clinic

Re: Nora Denise Stephenson
DOB: 14/03/78   Age: 47 years
NHS No: 576688058
Hospital No: 198013872

Dear Colleague,

I would be grateful if you could review Mrs Nora Stephenson in your outpatient knee clinic for further assessment and management of a suspected right medial meniscal tear.

Mrs Stephenson was admitted via A&E on 04/01/26 following a twisting injury to her right knee while stepping off a kerb earlier that day. She experienced immediate pain and swelling with difficulty weight-bearing. She reports pain predominantly over the medial aspect of the knee. There has been no true locking, no giving way, and no systemic symptoms.

Relevant background includes primary hypertension (diagnosed 2016, well controlled) and generalised anxiety and depression, stable on sertraline. She has a BMI of 29. There is no history of previous knee injury or surgery and no history of thromboembolic disease. She reports no drug allergies.

Usual medications are

## 4 - Add Augmentations to clinical Notes

This section allows for the augmentatio of clinical notes by:

- Replacing long phrases with abbreviations.
- Adding typos.
- Adding signatures.

In [13]:
augmentator = add_augmentations()
augmented_notes = await augmentator.run(True)
augmentator.write_final_documents_to_dataset()

Patient 0
Note: 0: Added (Estimated) 2 Abbreviations. Added 8 typos.
Note: 1: Added (Estimated) 7 Abbreviations. Added 17 typos.
Note: 2: Added (Estimated) 10 Abbreviations. Added 6 typos.
Note: 3: Error processing output 2: 'number_of_abbreviations'. Using original text.
Added (Estimated) 0 Abbreviations. Added 9 typos.
Note: 4: Added (Estimated) 48 Abbreviations. Added 19 typos.
Note: 5: Added (Estimated) 70 Abbreviations. Added 12 typos.
Note: 6: Error processing output 1: 'number_of_abbreviations'. Using original text.
Added (Estimated) 0 Abbreviations. Added 19 typos.
Note: 7: Error processing output 4: 'number_of_abbreviations'. Using original text.
Added (Estimated) 28 Abbreviations. Added 15 typos.
Note: 8: Added (Estimated) 7 Abbreviations. Added 9 typos.
Note: 9: Added (Estimated) 3 Abbreviations. Added 7 typos.
Note: 10: Added (Estimated) 27 Abbreviations. Added 7 typos.
Note: 11: Added (Estimated) 18 Abbreviations. Added 8 typos.
Note: 12: Added (Estimated) 3 Abbreviations.

## 5 - Write Clinical Notes to Dataset

- Writes the clinical notes to a dataset, alongside the patient journey and admission details.

In [ ]:
output_saver = save_final_outputs()
output_saver.run(run_name, current_time, version_tag)